#Landing para Bronze
Bronze layer com os dados 'as is ".

Nenhuma transformação feita, somente carregamento dos dados

In [0]:
catalog = "v_credit_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

In [0]:
%sql
USE CATALOG v_credit_medallion;
USE SCHEMA default;

-- TODO: List the files in the myfiles volume
LIST "/Volumes/v_credit_medallion/bronze/landing/";

In [0]:
files = _sqldf
file_names = [row['name'] for row in files.select('name').collect()]

In [0]:
from pyspark.sql.functions import current_timestamp
paths = {}
for file in file_names:
    name = (file[:-4]).lower()
    paths[name]=f"{landing_path}/{file}"
for file_name , path in paths.items():
    print(file_name)
    print(path)


In [0]:
df_dict = {}
for name , path in paths.items():
    df_dict[name] = spark.read.csv(path,header= True, inferSchema = True)
for dataset_name, dataset in df_dict.items():
    (dataset 
        .withColumn("ingestion_datetime", current_timestamp()) 
        .write.format("delta").mode("overwrite") 
        .saveAsTable(f"{catalog}.bronze.{dataset_name}")   
    )